# Week 3: Statistical Methods (ETS and ARIMA)
## In-Class Exercises

**Objective.** Fit the two workhorse statistical families, decide which one a series wants, and defend the choice with information criteria *and* out-of-sample error.

### How this notebook works

Three parts, each building on the one before it.

| Part | Format | Content |
| --- | --- | --- |
| 1 | Walkthrough | Holt-Winters, additive vs. multiplicative vs. damped. |
| 2 | Blanks we fill in together | Differencing and stationarity: ADF, KPSS, ACF, PACF. |
| 3 | On your own, ~12 min | Pick a SARIMA by hand from the ACF/PACF, then beat it (or not) with a search. |

In [ ]:
!pip install -q pandas numpy matplotlib statsmodels

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 4)

URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
air = pd.read_csv(URL, parse_dates=["Month"], index_col="Month")["Passengers"].asfreq("MS")

h, m = 24, 12
train, test = air[:-h], air[-h:]


def mae(y, yhat):
    return np.mean(np.abs(np.asarray(y, float) - np.asarray(yhat, float)))

def rmse(y, yhat):
    return np.sqrt(np.mean((np.asarray(y, float) - np.asarray(yhat, float)) ** 2))

def mase(y, yhat, y_train, m=12):
    y_train = np.asarray(y_train, float)
    return mae(y, yhat) / np.mean(np.abs(y_train[m:] - y_train[:-m]))

print(len(train), "train,", len(test), "test")

---
## Part 1. Holt-Winters, three ways

ETS is three equations, one each for level, trend, and season, each an exponentially weighted average of the last observation and the last estimate. Two modeling decisions:

- Seasonality **additive** (a fixed number of passengers each July) or **multiplicative** (a fixed *percentage*).
- Trend **damped** or not, which controls whether long-horizon forecasts flatten or extrapolate forever.

Week 2 showed this series is multiplicative. Watch what the additive fit does to the last test years.

In [ ]:
specs = {
    "ETS(A,A,A)":         dict(trend="add", seasonal="add", damped_trend=False),
    "ETS(A,A,M)":         dict(trend="add", seasonal="mul", damped_trend=False),
    "ETS(A,Ad,M) damped": dict(trend="add", seasonal="mul", damped_trend=True),
}

fits, fc = {}, pd.DataFrame(index=test.index)
for name, kw in specs.items():
    res = ExponentialSmoothing(train, seasonal_periods=m, initialization_method="estimated", **kw).fit()
    fits[name] = res
    fc[name] = res.forecast(h).to_numpy()

ax = train[-60:].plot(color="black", label="train")
test.plot(ax=ax, color="black", ls="--", label="test")
fc.plot(ax=ax)
ax.legend(fontsize=8)
ax.set_title("Three ETS specifications")
plt.show()

In [ ]:
tbl = pd.DataFrame({
    name: {"AIC": fits[name].aic,
           "MAE": mae(test, fc[name]),
           "RMSE": rmse(test, fc[name]),
           "MASE": mase(test, fc[name], train, m)}
    for name in fc
}).T
tbl.round(3).sort_values("MASE")

**Notice:**

1. The multiplicative fits track the widening summer peaks. The additive one runs low in the peaks and high in the troughs of the final years. That is the Week 2 lesson showing up as forecast error.
2. Damping barely matters over 24 months and matters a lot at 60. It is a statement about the *horizon* you care about.
3. AIC and out-of-sample MASE mostly agree here. When they disagree, believe the out-of-sample number and find out why the in-sample criterion was fooled.

In [ ]:
# Long-horizon view: where damping earns its keep.
long_fc = pd.DataFrame({name: fits[name].forecast(72).to_numpy()
                        for name in ["ETS(A,A,M)", "ETS(A,Ad,M) damped"]},
                       index=pd.date_range(train.index[-1] + pd.offsets.MonthBegin(), periods=72, freq="MS"))
ax = air.plot(color="black", label="actual")
long_fc.plot(ax=ax)
ax.set_title("72-month forecasts: damped vs. undamped trend")
plt.show()

---
## Part 2. Is it stationary yet?

ARIMA needs a stationary series: no trend, no seasonality, stable variance. Differencing gets you there. The two tests have opposite nulls on purpose.

- **ADF** null: there *is* a unit root (non-stationary). Small $p$ = stationary.
- **KPSS** null: the series *is* stationary. Small $p$ = non-stationary.

Reporting both protects you from the failure mode of each. Fill in the `TODO`s with me.

In [ ]:
def stationarity(y, label):
    y = pd.Series(y).dropna()
    adf_p = adfuller(y, autolag="AIC")[1]
    kpss_p = kpss(y, regression="c", nlags="auto")[1]
    verdict = "stationary" if (adf_p < 0.05 and kpss_p > 0.05) else "NOT stationary"
    return {"series": label, "ADF p": round(adf_p, 4), "KPSS p": round(kpss_p, 4), "verdict": verdict}


logair = np.log(train)

variants = {
    "log level":                 logair,
    "first difference":          logair.diff(),
    "seasonal difference (12)":  ...,   # TODO - hint: .diff(12)
    "both differences":          ...,   # TODO - seasonal difference, then a first difference
}

pd.DataFrame([stationarity(v, k) for k, v in variants.items() if not isinstance(v, type(Ellipsis))])

<details>
<summary><b>Show the two filled-in lines</b></summary>

```python
"seasonal difference (12)": logair.diff(12),
"both differences":        logair.diff(12).diff(),
```
</details>

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 8))
for row, (name, series) in enumerate([("log level", logair),
                                      ("diff(12)", logair.diff(12)),
                                      ("diff(12).diff()", logair.diff(12).diff())]):
    s = series.dropna()
    axes[row, 0].plot(s); axes[row, 0].set_title(name)
    plot_acf(s, lags=36, ax=axes[row, 1], title="ACF")
    plot_pacf(s, lags=36, ax=axes[row, 2], title="PACF", method="ywm")
plt.tight_layout()
plt.show()

**Reading the bottom row.** After both differences, $d = 1$, $D = 1$, $m = 12$. The ACF and PACF give the orders:

- A spike at **lag 1** in the ACF, cutting off after it, suggests $q = 1$.
- A spike at **lag 12**, with nothing at 24, suggests $Q = 1$.
- Decay in the PACF instead of the ACF would argue for AR terms.

That lands on $\text{ARIMA}(0,1,1)(0,1,1)_{12}$, the "airline model" Box and Jenkins made famous with this dataset.

---
## Part 3. Hand-picked vs. searched

About 12 minutes.

**Tasks.**

1. Fit $(0,1,1)(0,1,1)_{12}$ on `np.log(train)` with `SARIMAX`. Record AICc and test MASE (`np.exp` the forecast back).
2. Run the grid stub below over $p, q, P, Q \in \{0, 1\}$ with $d = D = 1$, and find the AICc minimum.
3. State whether the AICc winner also wins on MASE, and which you would ship.
4. Run Ljung-Box on the winner's residuals and report any structure left.

We forecast on the log scale and back-transform. `np.exp` of a mean forecast is a *median* on the original scale. Week 6 makes that matter.

In [ ]:
def fit_sarima(y_log, order, seasonal_order):
    return SARIMAX(y_log, order=order, seasonal_order=seasonal_order,
                   enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)


# Worked example of the scoring you will repeat:
res = fit_sarima(np.log(train), (0, 1, 1), (0, 1, 1, 12))
pred = np.exp(res.forecast(h))
print("AICc:", round(res.aicc, 2), " test MASE:", round(mase(test, pred, train, m), 3))

In [ ]:
# YOUR CODE HERE - task 2: the grid
# for p in [0, 1]:
#     for q in [0, 1]:
#         for P in [0, 1]:
#             for Q in [0, 1]:
#                 ...

<details>
<summary><b>Solution</b></summary>

```python
rows = []
for p in [0, 1]:
    for q in [0, 1]:
        for P in [0, 1]:
            for Q in [0, 1]:
                r = fit_sarima(np.log(train), (p, 1, q), (P, 1, Q, 12))
                pr = np.exp(r.forecast(h))
                rows.append({"order": f"({p},1,{q})({P},1,{Q})[12]",
                             "AICc": r.aicc,
                             "MASE": mase(test, pr, train, m)})

grid = pd.DataFrame(rows).sort_values("AICc")
print(grid.round(3).to_string(index=False))
print("\nAICc winner:", grid.iloc[0]["order"])
print("MASE winner:", grid.sort_values("MASE").iloc[0]["order"])

best = fit_sarima(np.log(train), (0, 1, 1), (0, 1, 1, 12))
print(acorr_ljungbox(best.resid[13:], lags=[12, 24], return_df=True))
best.plot_diagnostics(figsize=(11, 7)); plt.show()
```

**What you should find.** The airline model sits at or near the top on both criteria, which is satisfying and slightly rigged: this dataset is where the model came from. The lesson is in the disagreements. AICc is an in-sample complexity penalty, MASE an out-of-sample measurement. Large disagreements usually mean the test window contains something the training window did not.

For the automated version, `statsforecast`'s `AutoARIMA` (or `pmdarima.auto_arima`) runs the Hyndman-Khandakar stepwise search over a much larger space in about the same wall-clock time.
</details>

---
## Wrap-up

1. **ETS and ARIMA are different languages for the same thing.** ETS thinks in components, ARIMA in autocorrelation. Some ETS models have exact ARIMA equivalents.
2. **Differencing is ARIMA's price of admission.** Check with two tests that fail in opposite directions.
3. **AICc ranks in-sample, MASE measures out-of-sample.** Report both and investigate disagreements rather than picking the flattering one.
4. **Combine rather than select** when several models are close.

Next week: the same problem, handed to machine learning.